In [ ]:
from mdp import N_MDP, P_TWO_STATE, get_policy_grid, entropy, jensen_shannon, negative_entropy, negative_conditional_entropy
import torch as th
from einops import einsum
import numpy as np
import matplotlib.pyplot as plt
from mdp import TabularPolicy

policy_grid = get_policy_grid(100)

n_steps = 2000
lr = 0.01
beta = 0.001
mode = "nonlinear"

mosaic = """
    ABCE
    ABCE
"""
fig = plt.figure(layout="constrained", figsize=(15, 4))
ax = fig.subplot_mosaic(mosaic, gridspec_kw={"width_ratios": [1, 1, 1, 0.05]})

env = N_MDP(
    P=P_TWO_STATE, 
    f=lambda x: einsum(th.tensor([[0., 0.], [1., 1.]]), x, "s a, b s a -> b").sum(),
    gamma=0.85
)

heatmap = ax["A"].imshow(env.V(policy_grid).reshape(100, 100), origin='lower', extent=(0, 1, 0, 1))
bar = True
if bar:
    cbar = ax["A"].figure.colorbar(heatmap, ax=ax, cax=ax["E"])
    cbar.ax.set_ylabel("Utility", rotation=-90, va="bottom")

ax["A"].set_xlabel("$\pi(a_1|s_1)$")
ax["A"].set_ylabel("$\pi(a_1|s_2)$")
ax["A"].set_xticks([0, 1])
ax["A"].set_yticks([0, 1])
# ax.axis("off")
# ax.legend()

ax["B"].set_xticks([0, 1])
ax["B"].set_yticks([0, 1])
ax["B"].set_xlim([0, 1])
ax["B"].set_ylim([0, 1])
ax["B"].set_xticklabels(["$\delta_{s_1,a_1}$", "$\delta_{s_1,a_2}$"])
ax["B"].set_yticklabels(["$\delta_{s_2,a_1}$", "$\delta_{s_2,a_2}$"])
# ax.axis("off")
# ax.legend()


pi = TabularPolicy(env, 1, init_mode="second", beta=beta)
js = jensen_shannon(env.d_pi(pi.forward().detach()))
policies = pi.train(lr=lr, n_steps=n_steps)
pi = th.concatenate(policies)
ax["A"].plot(pi[:, 0, 0].detach(), pi[:, 0, 1].detach(), linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")

ax["C"].plot(range(len(pi.detach())), [env.f(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")

X, Y = env.d_pi(policy_grid)[:,0,0].reshape(100,100), env.d_pi(policy_grid)[:,1,0].reshape(100,100)
ax["B"].pcolormesh(X, Y, env.V(policy_grid).detach().numpy().reshape(100,100), cmap="viridis", shading='nearest')
ax["B"].plot(np.linspace(0, 1, 100), np.linspace(1, 0, 100), color="black", linestyle="--")
ax["B"].plot(env.d_pi(pi)[:, 0, 0].detach(), env.d_pi(pi)[:, 1, 0].detach(), linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")
#ax["B"].legend()

pi = TabularPolicy(env, 1, init_mode="second", beta=beta)
policies = pi.train(lr=lr, n_steps=n_steps, potential=negative_conditional_entropy)
pi = th.concatenate(policies)
ax["A"].plot(pi[:, 0, 0].detach(), pi[:, 0, 1].detach(), linewidth=0.8, ms=1, color="black", linestyle="dashed", label="NPG")

ax["C"].plot(range(len(pi.detach())), [env.f(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="dashed", label="NPG")

X, Y = env.d_pi(policy_grid)[:,0,0].reshape(100,100), env.d_pi(policy_grid)[:,1,0].reshape(100,100)
ax["B"].pcolormesh(X, Y, env.V(policy_grid).detach().numpy().reshape(100,100), cmap="viridis", shading='nearest')
ax["B"].plot(np.linspace(0, 1, 100), np.linspace(1, 0, 100), color="black", linestyle="--")
ax["B"].plot(env.d_pi(pi)[:, 0, 0].detach(), env.d_pi(pi)[:, 1, 0].detach(), linewidth=0.8, ms=1, color="black", linestyle="dashed", label="NPG")
#ax["B"].legend()

pi = TabularPolicy(env, 1, init_mode="second", beta=beta)
pi.env.g = lambda x: js(x)
policies = pi.train(lr=lr, n_steps=n_steps, potential=negative_entropy)
pi = th.concatenate(policies)
ax["A"].plot(pi[:, 0, 0].detach(), pi[:, 0, 1].detach(), linewidth=0.8, ms=1, color="black", linestyle="dotted", label="HPG")

ax["C"].plot(range(len(pi.detach())), [env.f(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="dotted", label="HPG")
#ax["C"].legend()

X, Y = env.d_pi(policy_grid)[:,0,0].reshape(100,100), env.d_pi(policy_grid)[:,1,0].reshape(100,100)
ax["B"].pcolormesh(X, Y, env.V(policy_grid).detach().numpy().reshape(100,100), cmap="viridis", shading='nearest')
ax["B"].plot(np.linspace(0, 1, 100), np.linspace(1, 0, 100), color="black", linestyle="--")
ax["B"].plot(env.d_pi(pi)[:, 0, 0].detach(), env.d_pi(pi)[:, 1, 0].detach(), linewidth=0.8, ms=1, color="black", linestyle="dotted", label="HPG")

ax["B"].legend()

from matplotlib.image import imread
im = imread("../manuscripts/gtml/graphics/kakades_example.png")
#Image box for solar pv logo
ax["B"].imshow(im, extent=(0.45, 0.95, 0.6, 0.75), zorder=10)

ax["A"].contour(env.g(env.d_pi(policy_grid)).reshape(100, 100), levels=[env.b], cmap="Reds_r", extent=(0, 1, 0, 1))
ax["B"].contour(X, Y, env.g(env.d_pi(policy_grid)).detach().numpy().reshape(100,100), levels=[env.b], cmap="Reds_r", extent=(0, 1, 0, 1))

ax["C"].set_xlabel("Policy Updates")
ax["C"].set_ylabel("Utility")
#ax["C"].set_title("Policy Return over Updates")

#plt.tight_layout()
plt.subplots_adjust(wspace=0.1)
plt.show()

plt.savefig("../manuscripts/gtml/graphics/pg_hpg_comparison.png", dpi=300)

In [ ]:
from mdp import N_MDP, P_TWO_STATE, get_policy_grid, entropy, jensen_shannon, negative_entropy, negative_conditional_entropy
import torch as th
from torch import einsum
import numpy as np
import matplotlib.pyplot as plt
from mdp import TabularPolicy

policy_grid = get_policy_grid(100)

n_steps = 2000
lr = 0.01
beta = 0.001
mode = "nonlinear"

mosaic = """
    ABCE
    ABCE
"""
fig = plt.figure(layout="constrained", figsize=(15, 4))
ax = fig.subplot_mosaic(mosaic, gridspec_kw={"width_ratios": [1, 1, 1, 0.05]})

env = N_MDP(
    P=P_TWO_STATE, 
    f=entropy,
    gamma=0.85
)

heatmap = ax["A"].imshow(env.V(policy_grid).reshape(100, 100), origin='lower', extent=(0, 1, 0, 1))
bar = True
if bar:
    cbar = ax["A"].figure.colorbar(heatmap, ax=ax, cax=ax["E"])
    cbar.ax.set_ylabel("Utility", rotation=-90, va="bottom")

ax["A"].set_xlabel("$\pi(a_1|s_1)$")
ax["A"].set_ylabel("$\pi(a_1|s_2)$")
ax["A"].set_xticks([0, 1])
ax["A"].set_yticks([0, 1])
# ax.axis("off")
# ax.legend()

ax["B"].set_xticks([0, 1])
ax["B"].set_yticks([0, 1])
ax["B"].set_xlim([0, 1])
ax["B"].set_ylim([0, 1])
ax["B"].set_xticklabels(["$\delta_{s_1,a_1}$", "$\delta_{s_1,a_2}$"])
ax["B"].set_yticklabels(["$\delta_{s_2,a_1}$", "$\delta_{s_2,a_2}$"])
# ax.axis("off")
# ax.legend()


pi = TabularPolicy(env, 1, init_mode="second", beta=beta)
js = jensen_shannon(env.d_pi(pi.forward().detach()))
policies = pi.train(lr=lr, lagrangian_lr=lr*10, n_steps=n_steps)
pi = th.concatenate(policies)
ax["A"].plot(pi[:, 0, 0].detach(), pi[:, 0, 1].detach(), linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")

ax["C"].plot(range(len(pi.detach())), [env.f(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")

X, Y = env.d_pi(policy_grid)[:,0,0].reshape(100,100), env.d_pi(policy_grid)[:,1,0].reshape(100,100)
ax["B"].pcolormesh(X, Y, env.V(policy_grid).detach().numpy().reshape(100,100), cmap="viridis", shading='nearest')
ax["B"].plot(np.linspace(0, 1, 100), np.linspace(1, 0, 100), color="black", linestyle="--")
ax["B"].plot(env.d_pi(pi)[:, 0, 0].detach(), env.d_pi(pi)[:, 1, 0].detach(), linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")
#ax["B"].legend()

pi = TabularPolicy(env, 1, init_mode="second", beta=beta)
policies = pi.train(lr=lr, n_steps=n_steps, potential=negative_conditional_entropy)
pi = th.concatenate(policies)
ax["A"].plot(pi[:, 0, 0].detach(), pi[:, 0, 1].detach(), linewidth=0.8, ms=1, color="black", linestyle="dashed", label="NPG")

ax["C"].plot(range(len(pi.detach())), [env.f(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="dashed", label="NPG")

X, Y = env.d_pi(policy_grid)[:,0,0].reshape(100,100), env.d_pi(policy_grid)[:,1,0].reshape(100,100)
ax["B"].pcolormesh(X, Y, env.V(policy_grid).detach().numpy().reshape(100,100), cmap="viridis", shading='nearest')
ax["B"].plot(np.linspace(0, 1, 100), np.linspace(1, 0, 100), color="black", linestyle="--")
ax["B"].plot(env.d_pi(pi)[:, 0, 0].detach(), env.d_pi(pi)[:, 1, 0].detach(), linewidth=0.8, ms=1, color="black", linestyle="dashed", label="NPG")
#ax["B"].legend()

pi = TabularPolicy(env, 1, init_mode="second", beta=beta)
pi.env.g = lambda x: js(x)
policies = pi.train(lr=lr, n_steps=n_steps, potential=negative_entropy)
pi = th.concatenate(policies)
ax["A"].plot(pi[:, 0, 0].detach(), pi[:, 0, 1].detach(), linewidth=0.8, ms=1, color="black", linestyle="dotted", label="HPG")

ax["C"].plot(range(len(pi.detach())), [env.f(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="dotted", label="HPG")
#ax["C"].legend()

X, Y = env.d_pi(policy_grid)[:,0,0].reshape(100,100), env.d_pi(policy_grid)[:,1,0].reshape(100,100)
ax["B"].pcolormesh(X, Y, env.V(policy_grid).detach().numpy().reshape(100,100), cmap="viridis", shading='nearest')
ax["B"].plot(np.linspace(0, 1, 100), np.linspace(1, 0, 100), color="black", linestyle="--")
ax["B"].plot(env.d_pi(pi)[:, 0, 0].detach(), env.d_pi(pi)[:, 1, 0].detach(), linewidth=0.8, ms=1, color="black", linestyle="dotted", label="HPG")

ax["B"].legend()

from matplotlib.image import imread
im = imread("../manuscripts/gtml/graphics/kakades_example.png")
#Image box for solar pv logo
ax["B"].imshow(im, extent=(0.45, 0.95, 0.6, 0.75), zorder=10)

ax["A"].contour(env.g(env.d_pi(policy_grid)).reshape(100, 100), levels=[env.b], cmap="Reds_r", extent=(0, 1, 0, 1))
ax["B"].contour(X, Y, env.g(env.d_pi(policy_grid)).detach().numpy().reshape(100,100), levels=[env.b], cmap="Reds_r", extent=(0, 1, 0, 1))

ax["C"].set_xlabel("Policy Updates")
ax["C"].set_ylabel("Utility")
#ax["C"].set_title("Policy Return over Updates")

#plt.tight_layout()
plt.subplots_adjust(wspace=0.1)
plt.show()

plt.savefig("../manuscripts/gtml/graphics/pg_hpg_comparison_nonlin2x2.png", dpi=300)

In [ ]:
from mdp import N_MDP, P_TWO_STATE, get_policy_grid, entropy, jensen_shannon, negative_entropy, negative_conditional_entropy
import torch as th
from torch import einsum
import numpy as np
import matplotlib.pyplot as plt
from mdp import TabularPolicy
import matplotlib as mpl

mpl.rcParams["axes.titlesize"] = 14.
mpl.rcParams["axes.labelsize"] = 14.
mpl.rcParams["axes.titleweight"] = "bold"
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams['text.usetex'] = True
mpl.rcParams['font.family'] = "serif"
mpl.rcParams['font.weight'] = "bold"

policy_grid = get_policy_grid(100)

n_steps = 1000
lr = 0.01
beta = 0.001
mode = "nonlinear"

mosaic = """
    ABCE
    ABDE
"""
fig = plt.figure(layout="constrained", figsize=(15, 4))
ax = fig.subplot_mosaic(mosaic, gridspec_kw={"width_ratios": [1, 1, 1, 0.05]})

env = N_MDP(
    P=P_TWO_STATE, 
    f=entropy,
    b=0.1,
    gamma=0.85
)

heatmap = ax["A"].imshow(env.V(policy_grid).reshape(100, 100), origin='lower', extent=(0, 1, 0, 1))
bar = True
if bar:
    cbar = ax["A"].figure.colorbar(heatmap, ax=ax, cax=ax["E"])
    cbar.ax.set_ylabel("Utility", rotation=-90, va="bottom")

ax["A"].set_xlabel("$\pi(a_1|s_1)$")
ax["A"].set_ylabel("$\pi(a_1|s_2)$")
ax["A"].set_xticks([0, 1])
ax["A"].set_yticks([0, 1])
# ax.axis("off")
# ax.legend()

ax["B"].set_xticks([0, 1])
ax["B"].set_yticks([0, 1])
ax["B"].set_xlim([0, 1])
ax["B"].set_ylim([0, 1])
ax["B"].set_xticklabels(["$\delta_{s_1,a_1}$", "$\delta_{s_1,a_2}$"])
ax["B"].set_yticklabels(["$\delta_{s_2,a_1}$", "$\delta_{s_2,a_2}$"])
# ax.axis("off")
# ax.legend()


pi = TabularPolicy(env, 1, init_mode="second", beta=beta)
js = jensen_shannon(env.d_pi(pi.forward().detach()))
pi.env.g = lambda x: js(x)
policies = pi.train(lr=lr*10, lagrangian_lr=lr*100, n_steps=n_steps)
pi = th.concatenate(policies)
ax["A"].plot(pi[:, 0, 0].detach(), pi[:, 0, 1].detach(), linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")

ax["C"].plot(range(len(pi.detach())), [env.f(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")
ax["D"].plot(range(len(pi.detach())), [env.g(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")

X, Y = env.d_pi(policy_grid)[:,0,0].reshape(100,100), env.d_pi(policy_grid)[:,1,0].reshape(100,100)
ax["B"].pcolormesh(X, Y, env.V(policy_grid).detach().numpy().reshape(100,100), cmap="viridis", shading='nearest')
ax["B"].plot(np.linspace(0, 1, 100), np.linspace(1, 0, 100), color="black", linestyle="--")
ax["B"].plot(env.d_pi(pi)[:, 0, 0].detach(), env.d_pi(pi)[:, 1, 0].detach(), linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")
#ax["B"].legend()

pi = TabularPolicy(env, 1, init_mode="second", beta=beta)
pi.env.g = lambda x: js(x)
policies = pi.train(lr=lr, n_steps=n_steps, potential=negative_conditional_entropy)
pi = th.concatenate(policies)
ax["A"].plot(pi[:, 0, 0].detach(), pi[:, 0, 1].detach(), linewidth=0.8, ms=1, color="black", linestyle="dashed", label="HPG")

ax["C"].plot(range(len(pi.detach())), [env.f(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="dashed", label="HPG")
ax["D"].plot(range(len(pi.detach())), [env.g(env.d_pi(p)).sum().detach() for p in policies], linewidth=0.8, ms=1, color="black", linestyle="dashed", label="HPG")


X, Y = env.d_pi(policy_grid)[:,0,0].reshape(100,100), env.d_pi(policy_grid)[:,1,0].reshape(100,100)
ax["B"].pcolormesh(X, Y, env.V(policy_grid).detach().numpy().reshape(100,100), cmap="viridis", shading='nearest')
ax["B"].plot(np.linspace(0, 1, 100), np.linspace(1, 0, 100), color="black", linestyle="--")
ax["B"].plot(env.d_pi(pi)[:, 0, 0].detach(), env.d_pi(pi)[:, 1, 0].detach(), linewidth=0.8, ms=1, color="black", linestyle="dashed", label="HPG")
#ax["B"].legend()

ax["B"].legend()

from matplotlib.image import imread
im = imread("../manuscripts/gtml/graphics/kakades_example.png")
#Image box for solar pv logo
ax["B"].imshow(im, extent=(0.45, 0.95, 0.6, 0.75), zorder=10)

ax["A"].contour(env.g(env.d_pi(policy_grid)).reshape(100, 100), levels=[env.b], cmap="Reds_r", extent=(0, 1, 0, 1))
ax["B"].contour(X, Y, env.g(env.d_pi(policy_grid)).detach().numpy().reshape(100,100), levels=[env.b], cmap="Reds_r", extent=(0, 1, 0, 1))

ax["C"].set_xlabel("Policy Updates")
ax["C"].set_ylabel("Utility")
ax["D"].set_xlabel("Policy Updates")
ax["D"].set_ylabel("Cost")
#ax["C"].set_title("Policy Return over Updates")

#plt.tight_layout()
plt.subplots_adjust(wspace=0.1)
#plt.show()

plt.savefig("../manuscripts/gtml/graphics/pg_hpg_comparison_nonlin2x2_constrained.png", dpi=300)

In [ ]:
from grid import draw_maze, plot_state_function
import matplotlib as mpl
from einops import rearrange
from mdp import EPSILON, p_grid_action, negative_entropy, entropy, mutual_information, jensen_shannon
from mdp import FiniteCMDP, TabularPolicy, N_MDP, plot_color_at, plot_pi_arrows
import torch as th
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams["axes.titlesize"] = 12.
mpl.rcParams["axes.labelsize"] = 12.
mpl.rcParams["axes.titleweight"] = "bold"
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams['text.usetex'] = True
mpl.rcParams['font.family'] = "serif"
mpl.rcParams['font.weight'] = "bold"

n_steps_vpg = 500
n_steps_hpo = 500

mosaic = """
    ABCDG
    EEFFH
"""
fig = plt.figure(layout="constrained", figsize=(10, 5))
axs = fig.subplot_mosaic(mosaic, gridspec_kw={"width_ratios": [1, 1, 1, 1, 0.05], "height_ratios": [1, 0.5]})

env = N_MDP(P=p_grid_action(5, 5), f=lambda x: mutual_information(x), b=0.1, gamma=0.9)
env.mu = th.zeros((25,))
env.mu[12] = 1.0  # start state

pi = TabularPolicy(env, 2, init_mode="uniform")
js = jensen_shannon(env.d_pi(pi.forward()).detach())
pi.env.g = lambda x: js(x) #+ entropy(x)

for i in range(1,4):
    for j in range(1,4):
        plot_color_at(pos=(i, j), ax=axs["A"], n=100, cmap="Greens")
axs["A"].set_title(r"Task: Constrained Diversity", fontsize=11)
for i in range(5):
    plot_color_at(pos=(4, i), ax=axs["A"], n=100, cmap="Reds")
    plot_color_at(pos=(0, i), ax=axs["A"], n=100, cmap="Reds")
    plot_color_at(pos=(i, 4), ax=axs["A"], n=100, cmap="Reds")
    plot_color_at(pos=(i, 0), ax=axs["A"], n=100, cmap="Reds")
axs["A"].axis("off")

axs["B"].set_title(r"$\pi_0$", fontsize=11)
plot_state_function(env.rho_pi(pi.forward()).detach()[0].reshape((5, 5)).numpy(), env.P.max(dim=-1).values, ax=axs["B"], n_tiles_per_state=100)
plot_pi_arrows(pi.forward().detach().numpy()[0], ax=axs["B"], n=100, s=5)

policies1 = pi.train(n_steps=n_steps_vpg, lr=0.1, potential=None, lagrangian_lr=1.)

pi = TabularPolicy(env, 2, init_mode="uniform")
js = jensen_shannon(env.d_pi(pi.forward()).detach())
pi.env.g = lambda x: js(x) #+ entropy(x)
policies2 = pi.train(n_steps=n_steps_hpo, lr=0.1, potential=negative_entropy)

axs["C"].set_title(r"$\pi_{T,1}$", fontsize=11)
plot_state_function(env.rho_pi(pi.forward()).detach()[0].reshape((5, 5)).numpy(), env.P.max(dim=-1).values, ax=axs["C"], n_tiles_per_state=100)
plot_pi_arrows(pi.forward().detach().numpy()[0], ax=axs["C"], n=100, s=5)

axs["D"].set_title(r"$\pi_{T,2}$", fontsize=11)
im = plot_state_function(env.rho_pi(pi.forward()).detach()[1].reshape((5, 5)).numpy(), env.P.max(dim=-1).values, ax=axs["D"], n_tiles_per_state=100)
plot_pi_arrows(pi.forward().detach().numpy()[1], ax=axs["D"], n=100, s=5)

bar = False
if bar:
    cbar = axs["D"].figure.colorbar(im, ax=axs["D"], cax=axs["G"])
    cbar.ax.set_ylabel("state occupancy", rotation=-90, va="bottom")
else:
    axs["G"].axis("off")

plt.savefig("../manuscripts/gtml/graphics/gridworld_cmdp.png", dpi=300, bbox_inches='tight')
axs["E"].plot(range(len(policies1)), [env.f(env.d_pi(p)).sum().detach() for p in policies1], linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")
axs["E"].plot(range(len(policies2)), [env.f(env.d_pi(p)).sum().detach() for p in policies2], linewidth=0.8, ms=1, color="black", linestyle="dotted", label="HPG")
axs["F"].plot(range(len(policies1)), [env.g(env.d_pi(p)).sum().detach() for p in policies1], linewidth=0.8, ms=1, color="black", linestyle="solid", label="VPG")
axs["F"].plot(range(len(policies2)), [env.g(env.d_pi(p)).sum().detach() for p in policies2], linewidth=0.8, ms=1, color="black", linestyle="dotted", label="HPG")
axs["F"].plot(range(len(policies1)), [env.b for _ in policies1], linewidth=0.8, ms=1, color="black", linestyle="dashed", label="limit")

axs["E"].set_xlabel("Policy Updates")
axs["E"].set_ylabel("Utility")
axs["F"].set_xlabel("Policy Updates")
axs["F"].set_ylabel("Cost")
axs["F"].legend()

axs["H"].axis("off")

plt.subplots_adjust(wspace=0.1, hspace=0.1)

plt.savefig("../manuscripts/gtml/graphics/gridworld_cmdp.png")

In [ ]:
from plots.grid import WilsonGridGraphGeneratorWLoops, max_pi
from plots.grid import draw_maze
import matplotlib.pyplot as plt
import numpy as np

_, axss = plt.subplots(3, 3)

for n, axs in zip((3, 5, 10), axss):
    pi_max = max_pi(n, n)
    pi_half = pi_max // 2

    for pi, ax in zip((0, pi_half, pi_max), axs):
        g = WilsonGridGraphGeneratorWLoops(n, n, pi=pi)
        a = np.array(g.generate_adjacency())

        draw_maze(n, n, a, ax)

plt.show()

In [ ]:
from plots.grid import WilsonGridGraphGeneratorWLoops, max_pi
from plots.grid import draw_maze
import matplotlib.pyplot as plt
import numpy as np

from plots.grid import cell_maze_from_adjacency

n = 5
g = WilsonGridGraphGeneratorWLoops(n, n, pi=max_pi(n, n) // 2)
a = np.array(g.generate_adjacency())
cells = cell_maze_from_adjacency(a, n)

masked_func = np.ma.masked_where(cells, np.ones_like(cells)).transpose()

_, ax = plt.subplots(1, 2)
ax[1].imshow(masked_func, cmap="Reds")

ax[1].tick_params(
        axis='both',  # changes apply to the x-axis
        which='both',  # both major and minor ticks are affected
        bottom=False,  # ticks along the bottom edge are off
        top=False,  # ticks along the top edge are off
        left=False,  # ticks along the bottom edge are off
        right=False,  # ticks along the top edge are off
        labelbottom=False,
        labelleft=False)  # labels along the bottom edge are off

ax[1].set_facecolor('grey')

draw_maze(n, n, a, ax[0])

In [ ]:
from mdp import FiniteCMDP, TabularPolicy, p_chain
import torch as th
import numpy as np
import matplotlib.pyplot as plt

env = FiniteCMDP(P=p_chain(10))
pi = TabularPolicy(env, 1, init_mode="right")
pi.env.mu = th.tensor([1.0, 0., 0., 0., 0., 0., 0., 0., 0., 0.])
pts = np.zeros((5, env.n_states))

mosaic = """
    AAAAAAAAAA
    BBBBBBBBBB
    BBBBBBBBBB
    BBBBBBBBBB
    BBBBBBBBBB
"""
fig = plt.figure(layout="constrained", figsize=(10, 5))
ax_dict = fig.subplot_mosaic(mosaic)

env.rho_pi(pi.forward()).detach().numpy()

import seaborn as sns
rho = env.rho_pi(pi.forward()).detach().numpy()

#sns.barplot(rho, ax=axs["A"], color="black")

pts[0] = rho

for t in range(4):
    pt = pi.belief_state(t).detach().numpy()
    pts[t+1] = pt * env.gamma**t

sns.heatmap([pts[0]], annot=False, fmt=".2f", cmap="Blues", ax=ax_dict["A"], cbar=False)
sns.heatmap(np.sqrt(pts[1:]), annot=False, fmt=".2f", cmap="Blues", cbar_kws={"label": "Probability"}, ax=ax_dict["B"])

ax_dict["A"].set_title(r"Discounted Occupancy $(1-\gamma) \sum_{t}\gamma^t p(s_t)$")
ax_dict["A"].axis("off")
ax_dict["B"].set_xlabel("State")
ax_dict["B"].set_ylabel("Time")
ax_dict["B"].set_title(r"Discounted Belief States $\gamma^t p(s_t)$ at Time $t$")
ax_dict["B"].set_xticks([])

#fig.tight_layout()
